#  <span style="color:orange">Binary Classification Tutorial（中文） (CLF102) - Level Intermediate</span>

**版本说明: PyCaret 2.2** <br />
**更新日期: 2022.07.31**

# 1.0 Tutorial Objective


欢迎来到二元分类教程 **(CLF102)** - 中级。本教程假设您已完成二元分类教程 (CLF101) - 初学者级别。如果您以前没有使用过 PyCaret 并且这是您的第一个教程，我们强烈建议您返回并继续阅读初学者教程，以了解使用 PyCaret 的基础知识。

在本教程中，我们将使用 pycaret.classification 模块来学习：

* **规范化**：如何规范化和缩放数据集
* **转换**：如何应用使数据呈线性和近似正态的转换
* **忽略低方差**：如何去除统计上不显着方差的特征以使实验更有效
* **删除多重共线性**：如何从数据集中删除多重共线性以提高线性算法的性能
* **组特征**：如何从数据集中的相关特征中提取统计信息
* **Bin Numeric Variables**：如何使用“sturges”规则对数值变量进行分类并将数值特征转换为分类特征
* **模型集成和堆叠**：如何使用多种集成技术（例如 Bagging、Boosting、软/硬投票和广义堆叠）来提高模型性能
模型校准：如何校准分类模型的概率
实验记录：如何使用 MLFlow 后端在 PyCaret 中记录实验

阅读时间：约 60 分钟

## 1.1 安装 PyCaret
如果您还没有安装 PyCaret，请按照[初学者教程](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)的链接获取有关如何安装的说明。

## 1.2 先决条件
Python 3.6 或更高版本
PyCaret 2.0 或更高版本
从 pycaret 的存储库加载数据的 Internet 连接
完成二元分类教程 (CLF101) - 初级
## 1.3 对于谷歌 colab 用户：
如果您在 Google colab 上运行此笔记本，请在笔记本顶部运行以下代码以显示交互式视觉效果。

`from pycaret.utils import enable_colab` <br/>
`enable_colab()`

## 1.4 See also:
- __[Binary Classification Tutorial (CLF101) - Level Beginner](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)__
- __[Binary Classification Tutorial (CLF103) - Level Expert](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Expert%20-%20CLF103.ipynb)__

# 2.0 本教程所涵盖技术的简要概述

在我们开始实际执行第 1 节中提到的技术之前，了解这些技术是什么以及何时使用它们是很重要的。大多数情况下，这些技术中的大多数将有助于线性和参数算法，但是在基于树的模型中也看到性能提升也就不足为奇了。下面的解释只是简短的，我们建议您做额外的阅读以更深入地了解并更彻底地了解这些技术。

- **标准化**：标准化/缩放（通常与标准化互换使用）用于转换数值变量的实际值，从而为机器学习提供有用的属性。许多算法，如逻辑回归、支持向量机、K 最近邻和朴素贝叶斯假设所有特征都以零为中心，并且方差处于相同的阶数。如果数据集中的某个特定特征的方差在数量级上大于其他特征，则模型可能无法正确理解所有特征并且可能表现不佳。例如，在我们用于此示例的数据集中，AGE 特征的范围在 21 到 79 之间，而其他数字特征的范围从 10,000 到 1,000,000。
__[阅读更多](https://sebastianraschka.com/Articles/2014_about_feature_scaling.html#z-score-standardization-or-min-max-scaling)__ <br/>
<br/>
- **变换**：虽然归一化变换数据范围以消除方差幅度的影响，但变换是一种更激进的技术，因为它改变了分布的形状，以便变换后的数据可以用正态或近似正态分布表示。一般来说，如果使用假设正态分布或高斯分布的算法，您应该转换数据。此类模型的示例是逻辑回归、线性判别分析 (LDA) 和高斯朴素贝叶斯。 （专业提示：名称中带有“Gaussian”的任何方法都可能假定正常。）__[阅读更多](https://en.wikipedia.org/wiki/Power_transform)__<br/>

- **忽略低方差**：数据集有时可能包含在样本中具有单个唯一值或少量值的分类特征。这种特征不仅不提供信息并且没有增加价值，而且有时对少数算法也是有害的。想象一个在样本中只有一个唯一值或少数占主导地位的唯一值的特征，可以通过使用 PyCaret 中的忽略低方差特征将它们从数据集中删除。

- **多重共线性**：多重共线性是数据集中独立特征之间具有非常高的相互关联或相互关联的状态。它是数据中的一种干扰，机器学习模型（主要是线性算法）无法很好地处理。多重共线性可能会降低模型的整体系数并导致不可预测的方差。这将导致过度拟合，模型可能在已知的训练集上表现出色，但在未知的测试集上会失败。 __[阅读更多](https://towardsdatascience.com/multicollinearity-in-data-science-c5f6c0fe6edf)__<br/>

- **组特征**：有时数据集可能包含在样本级别相关的特征。例如，在信用数据集中，有称为 BILL_AMT1 .. BILL_AMT6 的特征以这样的方式相关：BILL_AMT1 是 1 个月前的账单金额，BILL_AMT6 是 6 个月前的账单金额。此类特征可用于根据分布的统计特性（例如均值、中值、方差、标准差等）提取其他特征。

- **Bin 数值变量**：Binning 或离散化是将数值变量转换为分类特征的过程。一个例子是 Age 变量，它是数值的连续分布，可以离散化为间隔（10-20 岁、21-30 等）。分箱可以通过减少数据中的噪声或非线性来提高预测模型的准确性。 PyCaret 使用 Sturges 规则自动确定 bin 的数量和大小。__[阅读更多](https://www.vosesoftware.com/riskwiki/Sturgesrule.php)__<br/>
<br/>

- **模型集成和堆叠**：集成建模是一个创建多个不同模型以预测结果的过程。这可以通过使用许多不同的建模算法或使用不同的训练数据集样本来实现。然后，集成模型聚合每个基本模型的预测，从而对看不见的数据进行最终预测。使用集成模型的动机是减少预测的泛化误差。只要基础模型多样且独立，使用集成方法时模型的预测误差就会降低。集成学习中最常见的两种方法是 Bagging 和 Boosting。堆叠也是一种集成学习，其中来自多个模型的预测被用作预测最终结果的元模型的输入特征。__[阅读更多](https://blog.statsbot.co/ensemble-learning-d1dcd548e936)__<br/>


# 3.0 准备教程要用的数据集

在本教程中，我们将使用中使用的相同数据集__[二元分类教程 (CLF101) - 初学者级别](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)__

数据集致谢：
利奇曼，M.（2013 年）。 UCI 机器学习存储库。加利福尼亚州欧文：加利福尼亚大学信息与计算机科学学院。

原始数据集和数据字典可以在 UCI  __[found here](https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients)__ 机器学习存储库中找到。


# 4.0 获取数据

你可以从_[此处](https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients)找到的原始源下载数据并使用 pandas read_csv 函数加载它，或者您可以使用 PyCaret 的数据存储库使用 get_data 函数加载数据（这需要互联网连接）。

In [2]:
from pycaret.datasets import get_data
dataset = get_data('credit', profile=True)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

请注意，当 `profile` 参数为 `True` 时，它​​会显示用于探索性数据分析的数据配置文件。本实验将基于此分析执行上面第 2 节中讨论的几个预处理步骤。让我们总结一下配置文件如何帮助对数据做出关键的预处理选择。

- **缺失值**：数据中没有缺失值。但是，我们仍然需要管道中的插补器，以防新的未见数据有缺失值（在这种情况下不适用）。当您执行 setup() 函数时，会自动创建 imputers 并将其存储在管道中。默认情况下，它对数值使用均值估算器，对分类使用常数估算器。这可以使用 setup() 中的 numeric_imputation 和 categorical_imputation 参数进行更改。

- **多重共线性**：BILL_AMT1 ... BIL_AMT6 之间存在高度相关性，这将多重共线性引入数据。我们将通过在 setup 中使用 remove_multicollinearity 和 multicollinearity_threshold 参数来移除多重共线性。

- **数据比例/范围**：注意数字特征的比例/范围有何不同。例如，AGE 特征的范围在 21 到 79 之间，BILL_AMT1 的范围从 -165,580 到 964,511。这可能会导致假设所有特征在同一顺序内具有差异的算法出现问题。在这种情况下，BILL_AMT1 的数量级与 AGE 大不相同。我们将通过在 setup 中使用 normalize 参数来处理这个问题。 <br/>

- **特征空间的分布**：数字特征不是正态分布的。查看 `LIMIT_BAL、BILL_AMT1` 和 `PAY_AMT1 ... PAY_AMT6` 的分布。一些功能也高度倾斜，例如 PAY_AMT1。这可能会导致假设数据正态或近似正态分布的算法出现问题。示例包括逻辑回归、线性判别分析 (LDA) 和朴素贝叶斯。我们将通过在 setup.xml 中使用转换参数来处理这个问题。 <br/>

- **组特征**：从数据描述中我们知道某些特征是相互关联的，例如 `BILL_AMT1 ... BILL_AMT6` 和 `PAY_AMT1 ... PAY_AMT6`。我们将在设置中使用 group_features 参数从这些特征中提取统计信息。 <br/>

- **Bin Numeric Features**：在查看数字特征和目标变量之间的相关性时，我们认为 AGE 和 LIMIT_BAL 很弱。我们将使用 bin_numeric_features 参数从这些变量中去除噪声，这可能有助于线性算法。 <br/>


In [3]:
#check the shape of data
dataset.shape

(24000, 24)

为了在未见过的数据上演示 `predict_model()` 函数，已从原始数据集中保留了 1200 行的样本以用于预测。这不应与训练/测试拆分混淆，因为执行此特定拆分是为了模拟现实生活场景。另一种思考方式是，在执行机器学习实验时，这 1200 条记录不可用。

In [4]:
data = dataset.sample(frac=0.95, random_state=786)
data_unseen = dataset.drop(data.index)

data.reset_index(inplace=True, drop=True)
data_unseen.reset_index(inplace=True, drop=True)

print('Data for Modeling: ' + str(data.shape))
print('Unseen Data For Predictions ' + str(data_unseen.shape))

Data for Modeling: (22800, 24)
Unseen Data For Predictions (1200, 24)


# 5.0 在 PyCaret 中设置环境


在之前的教程 __[Binary Classification (CLF101) - Level Beginner](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)__ 中，我们学习了如何使用 setup() 在 pycaret 中初始化环境。在我们的上一个示例中没有传递其他参数，因为我们没有执行任何预处理步骤。在此示例中，我们将通过使用 setup() 自定义预处理管道将其提升到一个新的水平。让我们看看如何实现上面第 4 节中讨论的所有步骤。

In [5]:
from pycaret.classification import *

In [ ]:
exp_clf102 = setup(data = data, target = 'default', session_id=123,
                  normalize = True, 
                  transformation = True, 
                  ignore_low_variance = True,
                  remove_multicollinearity = True, multicollinearity_threshold = 0.95,
                  bin_numeric_features = ['LIMIT_BAL', 'AGE'],
                  group_features = [['BILL_AMT1', 'BILL_AMT2','BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6'],
                                   ['PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']],
                  log_experiment = True, experiment_name = 'credit1')

IntProgress(value=0, description='Processing: ', max=3)

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,20:13:52
Status,. . . . . . . . . . . . . . . . . .,Preprocessing Data


Text(value="Following data types have been inferred automatically, if they are correct press enter to continue…

,Data Type
LIMIT_BAL,Numeric
SEX,Categorical
EDUCATION,Categorical
MARRIAGE,Categorical
AGE,Numeric
PAY_1,Categorical
PAY_2,Categorical
PAY_3,Categorical
PAY_4,Categorical
PAY_5,Categorical



请注意，这与 [Binary Classification Tutorial (CLF101) - Level Beginner](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)中显示的设置网格相同。这里唯一的区别是传递给 setup() 的自定义参数现在设置为 True。另请注意，session_id 与初学者教程中使用的相同，这意味着随机化的效果是完全隔离的。我们在本实验中看到的任何改进都完全归功于 setup() 中采取的预处理步骤或本教程后面部分中使用的任何其他建模技术。

您可能已经注意到的另一个区别是我们在 setup 中使用的 log_experiment 和 Experiment_name 参数。这是为了记录本实验中的所有建模活动。您将在本笔记本的末尾看到如何从该功能中受益。

# 6.0 比较所有模型


与[Binary Classification Tutorial (CLF101) - Level Beginner](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)相似，我们还将使用 compare_models() 开始本教程。然后我们将下面的结果与上一个实验进行比较。

In [ ]:
top3 = compare_models(n_select = 3)

请注意，我们在 compare_models 中使用了 n_select 参数。在上一个教程中，您已经看到 compare_models 默认返回性能最佳的模型（基于默认排序顺序的单个模型）。但是，您可以使用 n_select 参数返回前 N 个模型。在此示例中 compare_models 已返回前 3 个模型。

In [ ]:
type(top3)

In [ ]:
print(top3)


为了进行比较，我们将使用 AUC 分数。请注意，在我们在 setup() 中执行预处理后，一些算法得到了极大的改进。

逻辑回归 AUC 从 0.6410 提高到 0.6957
朴素贝叶斯 AUC 从 0.6442 提高到 0.6724
K 最近邻 AUC 从 0.5939 提高到 0.6173
要查看上一教程中所有模型的结果，请[Binary Classification Tutorial (CLF101) - Level Beginner](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)中的第 7 节。

# 7.0 创建模型

在之前的教程 [Binary Classification (CLF101) - Level Beginner](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)中，我们学习了如何使用 create_model() 函数创建模型。现在我们将了解一些可能会派上用场的其他参数。在本节中，我们将使用 5 折交叉验证创建所有模型。注意 fold 参数是如何在 create_model() 中传递来实现这一点的。

### 7.1 创建模型（5折CV）

In [ ]:
dt = create_model('dt', fold = 5)

### 7.2 创建模型（四舍五入到小数点后两位）

In [ ]:
rf = create_model('rf', round = 2)

请注意在 `create_model()` 中传递 `round` 参数如何将评估指标四舍五入到小数点后两位。

# 8.0 调整模型


在之前的教程_[Binary Classification (CLF101) - Level Beginner](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)中，我们学习了如何使用预定义的网格自动调整模型的超参数。这里我们将介绍 tune_model() 中优化参数的使用，它可以被认为是一个目标函数。在 pycaret.classification 中，所有超参数调整都设置为默认优化准确性，可以使用优化参数进行更改。请参见下面的示例：

In [ ]:
tuned_rf = tune_model(rf, optimize = 'AUC')

In [ ]:
tuned_rf2 = tune_model(rf, optimize = 'Recall')

请注意调整随机森林分类器的结果与优化参数有何不同。在 tune_rf 中，我们优化了 AUC，导致 AUC 为 0.7691，召回率为 0.6446。但是，在 tune_rf2 中，当我们将优化参数设置为 Recall 时，它实际上在 Recall 方面产生了更好的模型。然而，AUC 受到了损害。观察下面tuned_rf和tuned_rf2的超参数的区别：

In [ ]:
plot_model(tuned_rf, plot = 'parameter')

In [ ]:
plot_model(tuned_rf2, plot = 'parameter')

# 9.0 集成模型


集成是一种常见的机器学习技术，用于提高模型的性能（主要是基于树的）。我们将在本节中介绍各种集成技术。其中包括 Bagging 和 Boosting[(阅读更多)](https://towardsdatascience.com/ensemble-methods-bagging-boosting-and-stacking-c9214a10a205)。我们将使用 PyCaret 中的 ensemble_model() 函数，它使用方法参数中定义的方法集成训练好的基本估计器。

In [ ]:
# lets create a simple decision tree model that we will use for ensembling 
dt = create_model('dt')

### 9.1 Bagging

In [ ]:
bagged_dt = ensemble_model(dt)

In [ ]:
# check the parameters of bagged_dt
print(bagged_dt)

请注意集成如何提高决策树分类器的 AUC。在上面的示例中，我们使用了使用 Bagging 方法的 ensemble_model() 的默认参数。让我们通过更改 ensemble_model() 中的方法参数来尝试 Boosting。请参见下面的示例：

### 9.2 Boosting

In [ ]:
boosted_dt = ensemble_model(dt, method = 'Boosting')

请注意在 PyCaret 中集成模型是多么容易。通过简单地更改方法参数，您可以进行装袋或提升，否则将需要多行代码。请注意， ensemble_model() 默认会构建 10 个估计器。这可以使用 n_estimators 参数进行更改。增加估计器的数量有时可以改善结果。请参阅下面的示例：

In [ ]:
bagged_dt2 = ensemble_model(dt, n_estimators=50)

请注意增加 n_estimators 参数如何提高集成模型的性能。具有默认 10 个估计量的 bagged_dt 模型的 AUC 为 0.7282，而在 n_estimators = 50 的 bagged_dt2 中，AUC 提高到 0.7503。

### 9.3 Blending


混合是可以在 PyCaret 中使用的另一种常用的集成技术。它使用来自多个模型的预测来生成最终的一组预测，使用来自在 estimator_list 参数中传递的所有模型的投票/多数共识。方法参数可用于定义投票的类型。当设置为硬时，它使用标签进行多数规则投票。当设置为软时，它使用预测概率的总和而不是标签。方法的默认值设置为自动，这意味着它会尝试使用软方法，如果不支持前者，则回退到硬方法。让我们看一个下面的例子：

In [ ]:
# train individual models to blend
lightgbm = create_model('lightgbm', verbose = False)
dt = create_model('dt', verbose = False)
lr = create_model('lr', verbose = False)

In [ ]:
# blend individual models
blend_soft = blend_models(estimator_list = [lightgbm, dt, lr], method = 'soft')

In [ ]:
# blend individual models
blend_hard = blend_models(estimator_list = [lightgbm, dt, lr], method = 'hard')

In [ ]:
# blend top3 models from compare_models
blender_top3 = blend_models(top3)

In [ ]:
print(blender_top3.estimators_)

### 9.4 Stacking


堆叠是另一种流行的集成技术，但由于实际困难而不太常见。 Stacking 是一种集成学习技术，它通过元模型组合多个模型。另一种考虑堆叠的方法是训练多个模型来预测结果，并创建一个元模型，将这些模型的预测与原始特征一起用作输入。 stack_models() 的实现基于 Wolpert, D. H. (1992b)。堆叠泛化（__[(阅读更多)](https://www.sciencedirect.com/science/article/abs/pii/S0893608005800231)__. ）。

让我们看一个使用我们从 compare_models 获得的前 3 个模型的示例：

In [ ]:
stack_soft = stack_models(top3)

In [ ]:
xgboost = create_model('xgboost')
stack_soft2 = stack_models(top3, meta_model=xgboost)

选择在堆叠中使用哪种方法和模型取决于数据集的统计属性。尝试不同的模型和方法是找出哪种配置最有效的最佳方式。然而，作为一般经验法则，具有强大而多样化性能的模型在用于堆叠时往往会改善结果。

在结束本节之前，stack_models() 中还有一个我们尚未见过的参数，称为 restack。此参数控制将原始数据公开给元模型的能力。当设置为 True 时，它​​将原始数据与基本模型的所有预测一起暴露给元模型。默认情况下，它设置为 True。

# 10.0 Model Calibration

在执行分类时，您通常不仅要预测类别标签（结果，例如 0 或 1），还要获得相应结果的概率，从而为预测提供一定程度的置信度。有些模型可以给你错误的类概率估计，有些甚至不支持概率预测。校准良好的分类器是概率性的，并以可以直接解释为置信水平的概率形式提供输出。 PyCaret 允许您通过 calibrate_model() 函数校准给定模型的概率。请参阅下面的示例：

In [ ]:
rf = create_model('rf')

In [ ]:
plot_model(rf, plot='calibration')

In [ ]:
calibrated_rf = calibrate_model(rf)

In [ ]:
plot_model(calibrated_rf, plot='calibration')

请注意上述 2 个图看起来有多么不同。一种是校准前，一种是校准后。完美校准的分类器将遵循上图中的黑色虚线。 calibrated_rf 不仅校准得更好，而且 AUC 也从 0.7591 提高到 0.7646。默认情况下，calibrate_model() 使用与 Platt 方法相对应的 sigmoid 方法。另一种可用的方法是等渗的，它是一种非参数方法。请参阅下面使用等渗方法进行校准的示例：

In [ ]:
calibrated_rf_isotonic = calibrate_model(rf, method = 'isotonic')

In [ ]:
plot_model(calibrated_rf_isotonic, plot='calibration')

# 11.0 实验日志

11.0 实验记录
PyCaret = 2.0 将 MLflow 跟踪组件作为后端 API 和 UI 嵌入，用于在运行机器学习代码时记录参数、代码版本、指标和输出文件，并在以后可视化结果。要在 pycaret 中记录您的实验，只需在 setup 函数中使用 log_experiment 和 Experiment_name 参数，就像我们在本例中所做的那样。

您可以在 https://localhost:5000 上启动 UI。只需从命令行或笔记本启动 MLFlow 服务器。请参见下面的示例：


In [ ]:
# to start the MLFlow server from notebook:
!mlflow ui 

### Open localhost:5000 on your browser (below is example of how UI looks like)
![title](https://i2.wp.com/pycaret.org/wp-content/uploads/2020/07/classification_mlflow_ui.png?resize=1080%2C508&ssl=1)

# 12.0 总结/下一步？

我们在本教程中介绍了许多新概念。最重要的是，我们已经看到了如何使用探索性数据分析来自定义 setup() 中的管道，与我们之前在_[二分类教程 (CLF101) - 初学者](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Beginner%20-%20%20CLF101.ipynb)中看到的相比，这大大改善了结果。我们还学习了如何在 PyCaret 中执行和调整集成。

然而，在 pycaret.classification 中有一些更高级的内容需要涵盖，包括定义和优化自定义成本函数、使用 shapley 值解释更复杂的基于树的模型、高级集成技术（如多层堆栈网络）和更多预处理管道方法。我们将在 pycaret.classification 系列的下一个也是最后一个教程中介绍所有这些。

下个教程见。按照链接到[二元分类教程 (CLF103) - 专家级别](https://github.com/pycaret/pycaret/blob/master/tutorials/Binary%20Classification%20Tutorial%20Level%20Expert%20-%20CLF103.ipynb)__